In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

from tyche import VolumeConfig, VolumeEstimator

/home/adam/.conda/envs/jax311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from tyche import ImplicitParamVector, ImplicitRandomVector, ImplicitVector
from tyche import print_gpu_memory

In [5]:
model = AutoModelForCausalLM.from_pretrained("EleutherAI/pythia-14m")
model.cuda()
tokenizer = AutoTokenizer.from_pretrained("EleutherAI/pythia-14m")
tokenizer.pad_token_id = 1  # pythia-specific
tokenizer.eos_token_id = 0  # pythia-specific
dataset = load_dataset("EleutherAI/lambada_openai", name="en", split="test", trust_remote_code=True)


The `GPTNeoXSdpaAttention` class is deprecated in favor of simply modifying the `config._attn_implementation`attribute of the `GPTNeoXAttention` class! It will be removed in v4.48


In [6]:
len(list(model.parameters()))

76

In [15]:

cfg = VolumeConfig(model=model, 
                   tokenizer=tokenizer, 
                   dataset=dataset, 
                   text_key="text",  # must match dataset field
                   n_samples=10,  # number of MC samples
                   cutoff=1e-2,  # KL-divergence cutoff (nats)
                   max_seq_len=8,  # sequence length for chunking dataset
                   val_size=10,  # number of sequences or chunks to use in estimation
                   data_batch_size=1,
                   cache_mode=None,
                   chunking=True,
                   implicit_vectors=False,
                   debug=False,
                   )
estimator = VolumeEstimator.from_config(cfg)

tokens.shape=torch.Size([52947, 8])


In [16]:
result = estimator.run()

  0%|          | 0/10 [00:00<?, ?it/s]

mults = tensor([1])
mults = tensor([1.], device='cuda:0')
mults = tensor([0.5000], device='cuda:0')
mults = tensor([0.7500], device='cuda:0')
mults = tensor([0.6250], device='cuda:0')


 10%|█         | 1/10 [00:00<00:03,  2.31it/s]

tensor([[0.6875]], device='cuda:0')
mults = tensor([1])
mults = tensor([1.], device='cuda:0')
mults = tensor([0.5000], device='cuda:0')
mults = tensor([0.7500], device='cuda:0')
mults = tensor([0.6250], device='cuda:0')


 20%|██        | 2/10 [00:00<00:03,  2.14it/s]

...

 60%|██████    | 6/10 [00:02<00:01,  2.39it/s]

tensor([[0.6250]], device='cuda:0')
mults = tensor([1])
mults = tensor([1.], device='cuda:0')


 60%|██████    | 6/10 [00:02<00:01,  2.02it/s]

mults = tensor([0.5000], device='cuda:0')
mults = tensor([0.7500], device='cuda:0')


KeyboardInterrupt: 

In [7]:
import gc

In [10]:
def list_largest_tensors():
    # Get all tensor objects
    tensors = []
    for obj in gc.get_objects():
        try:
            if torch.is_tensor(obj):
                tensors.append(obj)
        except:
            pass
    
    # Group tensors by memory location
    memory_dict = {}
    for t in tensors:
        if t.device.type == 'cuda':
            location = t.data_ptr()
            if location not in memory_dict:
                memory_dict[location] = []
            memory_dict[location].append(t)
    
    # Calculate sizes and sort by memory usage
    tensor_sizes = []
    for location, tensor_list in memory_dict.items():
        # Take the first tensor from each memory location
        tensor = tensor_list[0]
        size_mb = tensor.nelement() * tensor.element_size() / (1024 * 1024)
        tensor_sizes.append((size_mb, tensor.size(), tensor.dtype, len(tensor_list)))
    
    # Sort by size in descending order
    tensor_sizes.sort(reverse=True)
    
    # Calculate cumulative sizes relative to largest tensor
    if tensor_sizes:
        largest_size = tensor_sizes[0][0]
        cumulative = 0
    
    # Print results
    print(f"{'Size (MB)':>10} {'Cumul.(x)':>10} {'Shape':>20} {'Type':>10} {'Aliases':>8}")
    print("-" * 60)
    for size, shape, dtype, num_tensors in tensor_sizes:
        cumulative += size
        relative_cumul = cumulative / largest_size
        print(f"{size:10.2f} {relative_cumul:10.2f} {str(shape):>20} {str(dtype):>10} {num_tensors:>8}")

In [11]:
list_largest_tensors()

 Size (MB)  Cumul.(x)                Shape       Type  Aliases
------------------------------------------------------------
     53.66       1.00 torch.Size([14067712]) torch.float32        2
     24.56       1.46 torch.Size([50304, 128]) torch.float32        1
      4.00       1.53 torch.Size([1, 1, 2048, 2048]) torch.bool        1
      4.00       1.61 torch.Size([1, 1, 2048, 2048]) torch.bool        1
      4.00       1.68 torch.Size([1, 1, 2048, 2048]) torch.bool        1
      4.00       1.76 torch.Size([1, 1, 2048, 2048]) torch.bool        1
      4.00       1.83 torch.Size([1, 1, 2048, 2048]) torch.bool        1
      4.00       1.90 torch.Size([1, 1, 2048, 2048]) torch.bool        1
      0.25       1.91 torch.Size([512, 128]) torch.float32        1
      0.25       1.91 torch.Size([512, 128]) torch.float32        1
      0.25       1.92 torch.Size([512, 128]) torch.float32        1
      0.25       1.92 torch.Size([512, 128]) torch.float32        1
      0.25       1.93 torch.